In [ ]:
# Using VEP annotation without liftover

import polars as pl

# Config variables
dataset_subset = "WES3+WGS3_outer_join.rh_SPF_U42"
#dataset_subset = "U42_WES.all2"

dfs = []
for chrom in [str(num) for num in list(range(1, 21))]:
    #file = f"/master/abagwell/variant-analysis/results/rhesus/genotypes/annotated/U42_WES.all2.SNP.chr{chrom}.tsv"
    file = f"/master/abagwell/variant-analysis/results/rhesus/genotypes/annotated/{dataset_subset}.SNP.chr{chrom}.tsv"

    dfs.append(pl.read_csv(file, separator="\t", has_header=True, comment_prefix='##'))

df = pl.concat(dfs)

In [ ]:
df

In [ ]:
# Count consequences
consequences = df.with_columns(
# Separate out when there are multiple consequences for a variant
    pl.col("Consequence").str.split(",")
).explode("Consequence"
# Find count of each consequence
).group_by("Consequence").agg(pl.len()).sort("len").rename({
    'len': 'Count'
})
consequences

# consequences.write_csv(f"/master/abagwell/variant-analysis/results/rhesus/annotations/{dataset_subset}.SNP.consequences.tsv",
#     separator='\t')

In [ ]:
#df.unique("Location")

In [ ]:
# Count IMPACTs
df.with_columns(
    IMPACT = pl.col("Extra").str.split(";").list.first().str.split("=").list.last(),
).group_by("IMPACT").agg(pl.count('Location'))

In [ ]:
# Count IMPACT variants (this only counts one for each variant in the order of HIGH > MODERATE > LOW > MODIFIER)
impacts = df.with_columns(
    IMPACT = pl.col("Extra").str.split(";")#.list.eval(pl.element().str.contains("MODIFIER"))
).select("Location", "IMPACT").explode("IMPACT").filter(
    pl.col("IMPACT").str.contains("IMPACT")#.list.contains("IMPACT")
).with_columns(
    pl.col("IMPACT").str.split("=").list.last()
).with_columns(
    pl.col("IMPACT").str.replace("HIGH", "1_HIGH"),
).with_columns(
    pl.col("IMPACT").str.replace("MODERATE", "2_MODERATE"),
).with_columns(
    pl.col("IMPACT").str.replace("LOW", "3_LOW"),
).with_columns(
    pl.col("IMPACT").str.replace("MODIFIER", "4_MODIFIER"),
).group_by("Location").agg("IMPACT").with_columns(
    pl.col("IMPACT").list.sort()
).with_columns(
    pl.col("IMPACT").list.first()
).group_by("IMPACT").agg(pl.len("Loci")).sort("IMPACT").with_columns(
    pl.col("IMPACT").str.split("_").list.get(1)
)
impacts

In [ ]:
impacts.write_csv(f"/master/abagwell/variant-analysis/results/rhesus/annotations/{dataset_subset}.SNP.impacts.tsv",
    separator='\t')

In [ ]:
# impacts.with_columns(
#     pl.col("IMPACT").list.first()
# ).group_by("IMPACT").agg(pl.count("Location"))

In [ ]:
# import sgkit as sg
# from sgkit.io.vcf import vcf_to_zarr
# #chromosomes = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20"]
# chromosomes = ["19"]

# dfs = []
# for idx, chrom in enumerate(chromosomes):
#     # bcf = f"/master/abagwell/variant-analysis/results/rhesus/haplotypes/SHAPEIT5_merged/U42_WGS_WES.SNP.chr{chrom}.bcf"
#     # zarr = f"/master/abagwell/variant-analysis/results/rhesus/haplotypes/SHAPEIT5_merged/U42_WGS_WES.SNP.chr{chrom}.zarr"
#     bcf = f"/master/abagwell/variant-analysis/results/rhesus/genotypes/pass/U42_WES.all.SNP.chr{chrom}.bcf"
#     zarr = f"/master/abagwell/variant-analysis/results/rhesus/genotypes/pass/U42_WES.all.SNP.chr{chrom}.zarr"
#     try:
#         ds = sg.load_dataset(zarr)
#     except FileNotFoundError:
#         vcf_to_zarr(bcf, zarr)
#         ds = sg.load_dataset(zarr)

In [ ]:
import allel
import numpy as np

chromosomes = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20"]

# dfs = []
# for chrom in chromosomes:
#     vcf = f"/master/abagwell/variant-analysis/results/rhesus/genotypes/pass/{dataset_subset}.SNP.chr{chrom}.vcf.gz"
#     dfs.append(allel.read_vcf(vcf, ['variants/DP']))

vcf = "/master/abagwell/variant-analysis/results/rhesus/genotypes/pass/{dataset_subset}.SNP.autosomal.vcf.gz"
df = allel.read_vcf(vcf, ['variants/DP'])

In [ ]:
#np.concatenate([df['variants/DP'] for df in dfs]).mean() #/ 512

df['variants/DP'].mean()

In [ ]:
# Heterozygosity
chromosomes = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20"]

# dfs = []
# for chrom in chromosomes:
#     vcf = f"/master/abagwell/variant-analysis/results/rhesus/genotypes/pass/{dataset_subset}.SNP.chr{chrom}.vcf.gz"
#     dfs.append(allel.read_vcf(vcf, ['calldata/GT']))

vcf = "/master/abagwell/variant-analysis/results/rhesus/genotypes/pass/{dataset_subset}.SNP.autosomal.vcf.gz"
df = allel.read_vcf(vcf, ['calldata/GT'])

In [ ]:
#df = np.concatenate([df['calldata/GT'] for df in dfs])

In [ ]:
#callset = allel.read_vcf(input.vcf, ['calldata/GT'])
#g = allel.GenotypeArray(callset['calldata/GT'])

g = allel.GenotypeArray(df['calldata/GT'])
heterozygosity = allel.heterozygosity_observed(g)

In [ ]:
heterozygosity

In [ ]:
heterozygosity.mean()